# Bringing your own data

The earlier notebooks used `site.config()`, which fills in everything a
synthetic site already knows. With your own data you do that part yourself.
This notebook works from a file laid out like a FLUXNET2015 `FULLSET_HH`
file and covers:

1. reading the file and the four preparation steps every FLUXNET file needs;
2. mapping your columns to the drivers, including numbered soil sensors;
3. checking the time axis;
4. building, saving and reloading a configuration;
5. filling, with and without replacing values that arrived gap-filled;
6. the same fill from the command line, with the same numbers.

Nothing is downloaded. The file is written from the synthetic site at the
start, as a stand-in for a real one; with a real file, start at step 1. The
reference for all of this is [`docs/fluxnet.md`](../../docs/fluxnet.md). The
notebook takes about a minute to run.

In [1]:
import json
import shutil
import subprocess
import sys
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from rfrgapfill import (
    ColumnMap,
    FeatureConfig,
    RFRConfig,
    RFRGapFiller,
    RFRModel,
    load_config,
    prepare_time_index,
    synthetic_site,
)

# Show warnings as one line, without the path of the file that raised them.
warnings.formatwarning = lambda message, category, *args, **kwargs: (
    f"{category.__name__}: {message}\n"
)

workdir = Path(tempfile.mkdtemp(prefix="rfrgapfill-example-"))

### A stand-in file

The synthetic site, written the way FLUXNET writes its files: FLUXNET target
names, integer `TIMESTAMP_START`/`TIMESTAMP_END` stamps, `-9999` for missing
values, and soil variables numbered by sensor (`TS_F_MDS_1`, `SWC_F_MDS_1`).

In [2]:
site = synthetic_site()
stand_in = site.frame.rename(
    columns={
        "NEE": "NEE_VUT_REF",
        "NEE_QC": "NEE_VUT_REF_QC",
        "H": "H_F_MDS",
        "H_QC": "H_F_MDS_QC",
        "LE": "LE_F_MDS",
        "LE_QC": "LE_F_MDS_QC",
        "TS_F_MDS": "TS_F_MDS_1",
        "SWC_F_MDS": "SWC_F_MDS_1",
    }
)
stamps = stand_in.index
stand_in.insert(0, "TIMESTAMP_START", stamps.strftime("%Y%m%d%H%M").astype(np.int64))
stand_in.insert(
    1, "TIMESTAMP_END", (stamps + site.time_step).strftime("%Y%m%d%H%M").astype(np.int64)
)
source = workdir / "FLX_XX-Syn_FLUXNET2015_FULLSET_HH_2018-2018_1-4.csv"
stand_in.fillna(-9999).to_csv(source, index=False)
print("wrote", source.name)

wrote FLX_XX-Syn_FLUXNET2015_FULLSET_HH_2018-2018_1-4.csv


## 1. Read and prepare the file

Four steps, all of them ordinary pandas.

In [3]:
# pandas' default float parser can be one unit in the last place off. The
# forest is fitted on exact values, so read them exactly, as the command line does.
raw = pd.read_csv(source, float_precision="round_trip")

# 1. FLUXNET writes missing values as -9999. The package would read that as a
#    number, train on it and put it in the daily statistics.
raw = raw.replace(-9999, np.nan)

# 2. TIMESTAMP_START is an integer, YYYYMMDDHHMM. The package refuses numeric
#    timestamps, which pandas would read as nanoseconds since 1970.
raw.index = pd.to_datetime(raw["TIMESTAMP_START"].astype(str), format="%Y%m%d%H%M")

# 3. Rename the targets to the names the reporting layer knows. Default units,
#    the energy-balance check and the published benchmarks are keyed by NEE, H, LE.
raw = raw.rename(
    columns={
        "NEE_VUT_REF": "NEE",
        "NEE_VUT_REF_QC": "NEE_QC",
        "H_F_MDS": "H",
        "H_F_MDS_QC": "H_QC",
        "LE_F_MDS": "LE",
        "LE_F_MDS_QC": "LE_QC",
    }
)
raw.iloc[:3, :8]

,TIMESTAMP_START,TIMESTAMP_END,SW_IN_F,VPD_F_MDS,TA_F_MDS,NETRAD,WS,WD
TIMESTAMP_START,,,,,,,,
2018-01-01 00:00:00,201801010000,201801010030,0.0,1.392582,0.845334,-75.0,3.406974,169.576999
2018-01-01 00:30:00,201801010030,201801010100,0.0,1.701139,1.631854,-75.0,4.073614,163.870950
2018-01-01 01:00:00,201801010100,201801010130,0.0,1.704540,1.692241,-75.0,4.126017,156.243745


## 2. Map the drivers

Canonical names are internal; a `ColumnMap` says which of your columns is
which driver. `ColumnMap.fluxnet2015()` is the reference FLUXNET mapping, and
`missing_columns` checks it against the file before anything is fitted.

In [4]:
columns = ColumnMap.fluxnet2015()
print("missing from this file:", columns.missing_columns(raw.columns))

missing from this file: ('TS_F_MDS', 'SWC_F_MDS')


The paper names `TS_F_MDS` and `SWC_F_MDS` without saying which sensor. This
file numbers them, so choose explicitly. The choice goes into every run
manifest with the rest of the mapping.

In [5]:
columns = ColumnMap.fluxnet2015(
    overrides={"soil_temperature": "TS_F_MDS_1", "soil_water_content": "SWC_F_MDS_1"}
)
print("missing from this file:", columns.missing_columns(raw.columns))
pd.Series(dict(columns.variables), name="column").rename_axis("driver").to_frame()

missing from this file: ()


,column
driver,
shortwave,SW_IN_F
vpd,VPD_F_MDS
air_temperature,TA_F_MDS
net_radiation,NETRAD
wind_speed,WS
wind_direction,WD
soil_heat_flux,G_F_MDS
soil_temperature,TS_F_MDS_1
relative_humidity,RH


## 3. Check the time axis

Every duration in the package is elapsed time, never a row count: one day is
48 rows only at 30-minute cadence with no missing rows. `prepare_time_index`
validates and sorts the timestamps, refuses duplicates unless you choose how
to resolve them, and infers the cadence from the real differences.

In [6]:
prepared, axis = prepare_time_index(raw)
print("cadence:", axis.time_step, "| regular:", axis.is_regular, f"| coverage: {axis.coverage:.1%}")
print("rows in a 7-day gap at this cadence:", axis.periods("7d"))
axis.to_dict()

cadence: 0:30:00 | regular: True | coverage: 100.0%
rows in a 7-day gap at this cadence: 336


{'start': '2018-01-01T00:00:00',
 'end': '2018-12-31T23:30:00',
 'timezone': None,
 'n_timestamps': 17520,
 'time_step': 'P0DT0H30M0S',
 'time_step_source': 'inferred',
 'is_regular': True,
 'n_expected': 17520,
 'n_missing': 0,
 'n_off_grid': 0,
 'n_duplicates_removed': 0,
 'coverage': 1.0}

## 4. A configuration of your own

`RFRConfig` validates every setting as it is built. A latitude (or
`hemisphere=`) is required, because it decides the season feature. Every
scientific choice is a field, and `to_dict()` is what goes into the run
manifest.

In [7]:
config = RFRConfig(
    mode="RFR10",
    frequency="30min",
    latitude=45.0,
    site_id="XX-Syn",
    random_state=42,
    column_map=columns,
    features=FeatureConfig(daily_statistic_strategy="rolling_available"),  # A4
    hyperparameter_grid={"n_estimators": (50,)},  # one small point, for speed (A1)
    n_jobs=-1,
)
print("paper faithful:", config.is_paper_faithful)

paper faithful: True


Saved as JSON, the same mapping is a configuration file for `load_config`
and the command line. Only `mode` and what the constructor demands are
required in a hand-written file; every other setting takes its documented
default, and a misspelt key is refused rather than ignored.

In [8]:
config_path = workdir / "XX-Syn.json"
config_path.write_text(json.dumps(config.to_dict(), indent=2), encoding="utf-8")
reloaded = load_config(config_path)
print("round trip identical:", reloaded.to_dict() == config.to_dict())
print(json.dumps(config.to_dict()["features"], indent=2)[:400], "...")

round trip identical: True
{
  "use_receptive_limiter": true,
  "feature_mode": "paper_safe",
  "radiation_thresholds": [
    10.0,
    100.0
  ],
  "boundary_convention": "medium_inclusive",
  "min_daily_observations": 1,
  "daily_statistic_strategy": "rolling_available",
  "fallback_window_days": 7,
  "daily_std_ddof": 1
} ...


## 5. Fill

FLUXNET's target columns arrive already gap-filled, flagged by their QC
columns. By default those values are carried through and labelled
`pre_filled`; only rows with no value at all are predicted.
`refill_pre_filled=True` replaces them deliberately with RFR predictions, and
the originals stay in `LE_original`.

In [9]:
filler = RFRGapFiller(config).fit(raw, target="LE", qc_column="LE_QC")
default_fill = filler.fill(raw)
refill = filler.fill(raw, refill_pre_filled=True)

print(default_fill.report.summary())
print(refill.report.summary())
pd.DataFrame(
    {
        "default": default_fill.fill_method.value_counts(),
        "refill_pre_filled=True": refill.fill_method.value_counts(),
    }
).fillna(0).astype(int)

LE: filled 849 of 849 candidate row(s) with RFR10.
LE: filled 2390 of 2390 candidate row(s) with RFR10.


,default,refill_pre_filled=True
LE_fill_method,,
RFR10,849,2390
observed,15130,15130
pre_filled,1541,0


The fitted model can be saved and reloaded on its own. Its configuration is
validated again as it loads, and the feature order it was fitted on travels
with it.

In [10]:
model_path = workdir / "LE.joblib"
filler.model.save(model_path)
model = RFRModel.load(model_path)
print("features:", model.get_feature_names())
print("best parameters:", model.get_best_params())

features: ('shortwave', 'vpd', 'air_temperature', 'net_radiation', 'wind_speed', 'wind_direction', 'soil_heat_flux', 'soil_temperature', 'relative_humidity', 'soil_water_content', 'radiation_category', 'time_distance_hours', 'season', 'LE_daily_q1', 'LE_daily_q2', 'LE_daily_q3', 'LE_daily_std')
best parameters: {'n_estimators': 50}


## 6. The same fill from the command line

`rfr-gapfill` is a thin shell over the same API. `--na-value` and
`--timestamp-format` do steps 1 and 2 above; the targets keep their FLUXNET
names. In a terminal the command is:

```bash
rfr-gapfill fill FLX_XX-Syn_FLUXNET2015_FULLSET_HH_2018-2018_1-4.csv \
  --config XX-Syn.json --target LE_F_MDS --qc-column LE_F_MDS_QC \
  --timestamp TIMESTAMP_START --timestamp-format %Y%m%d%H%M --na-value -9999 \
  --output XX-Syn_LE_filled.csv
```

Here it runs through the notebook's own interpreter so it works in any
environment the package is installed in.

In [11]:
output = workdir / "XX-Syn_LE_filled.csv"
command = [
    sys.executable,
    "-m",
    "rfrgapfill.cli",
    "fill",
    str(source),
    "--config",
    str(config_path),
    "--target",
    "LE_F_MDS",
    "--qc-column",
    "LE_F_MDS_QC",
    "--timestamp",
    "TIMESTAMP_START",
    "--timestamp-format",
    "%Y%m%d%H%M",
    "--na-value",
    "-9999",
    "--output",
    str(output),
]
completed = subprocess.run(command, capture_output=True, text=True, check=False)
print("exit status:", completed.returncode)
print(completed.stdout.replace(str(workdir), "<workdir>").strip())
print("wrote:", sorted(path.name for path in workdir.iterdir()))

exit status: 0
LE_F_MDS: filled 849 of 849 candidate row(s) with RFR10.
wrote <workdir>\XX-Syn_LE_filled.csv
wrote <workdir>\XX-Syn_LE_filled.manifest.json
wrote: ['FLX_XX-Syn_FLUXNET2015_FULLSET_HH_2018-2018_1-4.csv', 'LE.joblib', 'XX-Syn.json', 'XX-Syn_LE_filled.csv', 'XX-Syn_LE_filled.manifest.json']


The command line wrote the filled series and a manifest beside it. Its
values match the notebook's fill:

In [12]:
from_cli = pd.read_csv(output, index_col=0, parse_dates=True)["LE_F_MDS_filled"]
from_api = default_fill.filled
print("identical values:", np.allclose(from_cli.to_numpy(), from_api.to_numpy(), equal_nan=True))

identical values: True


In [13]:
shutil.rmtree(workdir)